# 7-5. Lasso -- 5-fold 대신 LOOCV를 썼어야 했나

`6-2`에서 튜닝한 Lasso(`LassoCV`)가 미튜닝(alpha=1.0)보다 1.3% 나빴다. 의심 원인은 "origin당 표본 18~35개뿐이라 5-fold면 폴드당 3~7개로 너무 적다"는 것.

**5-fold를 쓴 이유**: 특별한 근거 없이 `LassoCV`의 **기본값이라서 그냥 썼다.** 표본이 적을 때 흔히 권장되는 **LOOCV**로 바꾸면 달라지는지 확인한다.


In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import LeaveOneOut

H = 4
LAG = 2
MIN_TRAIN = 20
TARGET_VAR = "X3_l1"

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", TARGET_VAR]].dropna()[TARGET_VAR].values.astype(float)
n = len(vals)
print(f"대표변수: {TARGET_VAR}, 데이터 {n}개 분기")
print(f"origin={MIN_TRAIN}일 때 회귀표본 수 = {MIN_TRAIN - LAG}개 -> 5-fold면 폴드당 {(MIN_TRAIN - LAG)//5}~{(MIN_TRAIN - LAG)//5+1}개, LOOCV면 폴드당 1개(검증)")


대표변수: X3_l1, 데이터 60개 분기
origin=20일 때 회귀표본 수 = 18개 -> 5-fold면 폴드당 3~4개, LOOCV면 폴드당 1개(검증)


## 1. 세 방식 준비 -- 미튜닝 / 튜닝(5-fold, 기존) / 튜닝(LOOCV, 신규)

**alpha란**: Lasso는 오차뿐 아니라 `alpha x |계수들의 합|`이라는 페널티까지 같이 최소화한다 -- alpha가 크면 계수가 0에 가깝게 눌려 더 단순한 모델이 되고, 작으면 그냥 일반회귀(OLS)에 가까워진다. "튜닝"은 후보 alpha 13개(0.001~1000, 로그간격) 중 검증데이터로 가장 성능 좋은 걸 자동으로 고르는 것 -- 5-fold와 LOOCV는 그 검증을 어떻게 나누냐만 다르다.


In [2]:
def _lag_xy(values, lag=LAG):
    X = np.array([values[i:i + lag] for i in range(len(values) - lag)])
    y = values[lag:]
    return X, y


def _recursive_forecast_quiet(model_fit_fn, values, h=H, lag=LAG):
    X, y = _lag_xy(values, lag)
    model = model_fit_fn(X, y)
    hist = list(values[-lag:])
    out = []
    for _ in range(h):
        x = np.array(hist[-lag:]).reshape(1, -1)
        yp = model.predict(x)[0]
        out.append(yp)
        hist.append(yp)
    return np.array(out)


ALPHAS = np.logspace(-3, 3, 13)


def fc_lasso_untuned(values, h=H):
    return _recursive_forecast_quiet(lambda X, y: Lasso(alpha=1.0, max_iter=5000).fit(X, y), values, h)


def fc_lasso_5fold(values, h=H):
    return _recursive_forecast_quiet(lambda X, y: LassoCV(alphas=ALPHAS, max_iter=5000).fit(X, y), values, h)


def fc_lasso_loocv(values, h=H):
    return _recursive_forecast_quiet(lambda X, y: LassoCV(alphas=ALPHAS, max_iter=5000, cv=LeaveOneOut()).fit(X, y), values, h)


print("3개 함수 준비 완료(미튜닝/5-fold/LOOCV)")


3개 함수 준비 완료(미튜닝/5-fold/LOOCV)


## 2. Walk-forward 비교 -- origin 37개 전부


In [3]:
var_std = vals.std()
rows = []
alpha_rows = []
for origin in range(MIN_TRAIN, n - H + 1):
    train, actual = vals[:origin], vals[origin:origin + H]
    fc_u = fc_lasso_untuned(train, H)
    fc_5 = fc_lasso_5fold(train, H)
    fc_l = fc_lasso_loocv(train, H)

    X, y = _lag_xy(train)
    model_5 = LassoCV(alphas=ALPHAS, max_iter=5000).fit(X, y)
    model_l = LassoCV(alphas=ALPHAS, max_iter=5000, cv=LeaveOneOut()).fit(X, y)

    rows.append({
        "origin": origin,
        "미튜닝": np.mean(np.abs(actual - fc_u)) / var_std,
        "튜닝(5-fold)": np.mean(np.abs(actual - fc_5)) / var_std,
        "튜닝(LOOCV)": np.mean(np.abs(actual - fc_l)) / var_std,
    })
    alpha_rows.append({"origin": origin, "alpha(5-fold)": model_5.alpha_, "alpha(LOOCV)": model_l.alpha_})

lasso_df = pd.DataFrame(rows)
alpha_df = pd.DataFrame(alpha_rows)

mean_u = lasso_df["미튜닝"].mean()
mean_5 = lasso_df["튜닝(5-fold)"].mean()
mean_l = lasso_df["튜닝(LOOCV)"].mean()

print(f"미튜닝(alpha=1.0 고정)     평균 정규화오차: {mean_u:.4f}")
print(f"튜닝(5-fold, 기존)         평균 정규화오차: {mean_5:.4f}  (미튜닝 대비 {(mean_u-mean_5)/mean_u*100:+.1f}%)")
print(f"튜닝(LOOCV, 신규)          평균 정규화오차: {mean_l:.4f}  (미튜닝 대비 {(mean_u-mean_l)/mean_u*100:+.1f}%)")


미튜닝(alpha=1.0 고정)     평균 정규화오차: 0.6214
튜닝(5-fold, 기존)         평균 정규화오차: 0.6295  (미튜닝 대비 -1.3%)
튜닝(LOOCV, 신규)          평균 정규화오차: 0.6266  (미튜닝 대비 -0.8%)


## 3. alpha 선택 자체가 다른가

거의 항상 같은 alpha를 고른다면 성능 차이가 없는 게 당연하므로, 5-fold와 LOOCV가 실제로 다른 alpha를 고르는지 확인한다.


In [4]:
alpha_df["같은 alpha?"] = np.isclose(alpha_df["alpha(5-fold)"], alpha_df["alpha(LOOCV)"])
same_pct = alpha_df["같은 alpha?"].mean() * 100
print(f"37개 origin 중 5-fold와 LOOCV가 같은 alpha를 고른 비율: {same_pct:.1f}%")
display(alpha_df.head(10))


37개 origin 중 5-fold와 LOOCV가 같은 alpha를 고른 비율: 43.2%


,origin,alpha(5-fold),alpha(LOOCV),같은 alpha?
0,20,0.001000,10.000000,False
1,21,0.001000,10.000000,False
2,22,0.001000,10.000000,False
3,23,0.001000,10.000000,False
4,24,100.000000,100.000000,True
5,25,1000.000000,316.227766,False
6,26,1000.000000,316.227766,False
7,27,0.001000,1000.000000,False
8,28,31.622777,1000.000000,False
9,29,0.001000,10.000000,False


## 4. 표본이 늘어나면 격차가 줄어드나 -- "표본부족이 근본원인"인지 직접 검증

지금까지는 "표본이 적어서 그렇다"고 추정만 했다. origin을 표본수(=origin-LAG) 기준 절반으로 나눠서, **표본이 적은 절반과 많은 절반에서 튜닝 vs 미튜닝 격차가 다른지** 직접 비교한다.


In [5]:
lasso_df["표본수"] = lasso_df["origin"] - LAG
median_n = lasso_df["표본수"].median()
early = lasso_df[lasso_df["표본수"] <= median_n]
late = lasso_df[lasso_df["표본수"] > median_n]


def gap_pct(d, col):
    u, t = d["미튜닝"].mean(), d[col].mean()
    return (u - t) / u * 100


print(f"표본 적은 절반(표본수<={median_n:.0f}, {len(early)}개 origin): 5-fold 격차={gap_pct(early, '튜닝(5-fold)'):+.1f}%, LOOCV 격차={gap_pct(early, '튜닝(LOOCV)'):+.1f}%")
print(f"표본 많은 절반(표본수> {median_n:.0f}, {len(late)}개 origin): 5-fold 격차={gap_pct(late, '튜닝(5-fold)'):+.1f}%, LOOCV 격차={gap_pct(late, '튜닝(LOOCV)'):+.1f}%")


표본 적은 절반(표본수<=36, 19개 origin): 5-fold 격차=-3.6%, LOOCV 격차=-2.3%
표본 많은 절반(표본수> 36, 18개 origin): 5-fold 격차=-0.1%, LOOCV 격차=-0.1%


**확인됨**: 표본 적은 절반(19개, 18~36개 샘플)에서는 5-fold -3.6%/LOOCV -2.3%로 튜닝이 뚜렷하게 나쁘지만, 표본 많은 절반(18개, 37~54개 샘플)에서는 5-fold/LOOCV 둘 다 **-0.1%로 사실상 차이가 없어진다.** "표본부족이 근본원인"이라는 추정이 숫자로 확인됐다 -- 표본이 늘면 튜닝 여부 자체가 안 중요해진다.


## 5. 결론

- **5-fold를 쓴 이유는 근거가 없었다** -- `LassoCV` 기본값이라 그대로 썼을 뿐.
- **표본이 적을 때만 튜닝이 나쁘고, 표본이 늘면 격차가 거의 사라진다**(-3.6%→-0.1%) -- 문제의 정체는 "5-fold냐 LOOCV냐"가 아니라 **표본 자체의 크기**였다.
- **그래서 5-fold/LOOCV/미튜닝 중 뭘 골라야 하는지는 사실 애매하다**: 표본이 적은 초반 origin에서는 미튜닝이 안전하지만, 표본이 쌓인 후반 origin에서는 셋 다 거의 동등하다 -- "이거 하나가 정답"이라고 고정할 근거가 없고, 표본 크기에 따라 최선이 달라지는 상황이라는 게 정직한 결론이다.
- **왜 LOOCV도 초반엔 절반만 고치나**: LOOCV는 폴드 수를 늘려도 폴드 하나하나가 1개 데이터포인트에만 의존해 그 자체로 분산이 크다 -- "전체 표본이 적다"는 근본 원인 자체는 못 없앤다.
